# Quickstart: Querying PDF With Astra and LangChain

### A question-answering demo using Astra DB and LangChain, powered by Vector Search

#### Pre-requisites:

You need a **_Serverless Cassandra with Vector Search_** database on [Astra DB](https://astra.datastax.com) to run this demo. As outlined in more detail [here](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html#_prepare_for_using_your_vector_database), you should get a DB Token with role _Database Administrator_ and copy your Database ID: these connection parameters are needed momentarily.

You also need an [OpenAI API Key](https://cassio.org/start_here/#llm-access) for this demo to work.

#### What you will do:

- Setup: import dependencies, provide secrets, create the LangChain vector store;
- Run a Question-Answering loop retrieving the relevant headlines and having an LLM construct the answer.

Import the packages you'll need:

In [ ]:
# ============================================================
# STEP 3 - IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from pypdf import PdfReader

from langchain_huggingface import (
    HuggingFaceInferenceAPIEmbeddings,
    HuggingFaceEndpoint,
    ChatHuggingFace,
)

from langchain_astradb import AstraDBVectorStore

from langchain_text_splitters import CharacterTextSplitter

c:\Data Science 2026\Project Repository\AI_Agents_LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================
# STEP 4 - ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

ASTRA_DB_APPLICATION_TOKEN = os.getenv(
    "ASTRA_DB_APPLICATION_TOKEN"
)

ASTRA_DB_API_ENDPOINT = os.getenv(
    "ASTRA_DB_API_ENDPOINT"
)

print("HF token loaded:", bool(HF_TOKEN))
print("Astra token loaded:", bool(ASTRA_DB_APPLICATION_TOKEN))
print("Astra endpoint loaded:", bool(ASTRA_DB_API_ENDPOINT))

Astra token loaded: True
Astra endpoint loaded: True


In [ ]:


# ============================================================
# STEP 5 - READ PDF
# ============================================================

pdfreader = PdfReader("C:\\Data Science 2026\\Project Repository\\AI_Agents_LangGraph\\10.PDFQuery_LangChain\\Budget_Speech.pdf")

raw_text = ""

for page in pdfreader.pages:

    content = page.extract_text()

    if content:
        raw_text += content + "\n"

print(f"Extracted {len(raw_text)} characters")

Extracted 92141 characters


In [ ]:
print(raw_text[:3000])

In [ ]:
# ============================================================
# STEP 6 - TEXT SPLITTING
# ============================================================

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=800,
    chunk_overlap=200,
    length_function=len,
)

texts = text_splitter.split_text(raw_text)

print(f"Created {len(texts)} chunks")

Created 153 chunks


In [ ]:
# ============================================================
# STEP 7 - HUGGING FACE CLOUD EMBEDDINGS
# ============================================================

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceInferenceAPIEmbeddings(
    api_key=HF_TOKEN,
    model_name=embedding_model,
)

print("Hugging Face cloud embedding model configured")

Embedding model loaded


In [ ]:
# ============================================================
# STEP 8 - TEST CLOUD EMBEDDINGS
# ============================================================

test_embedding = embeddings.embed_query(
    "What is India's GDP?"
)

print("Embedding dimensions:", len(test_embedding))
print(test_embedding[:10])

Embedding dimensions: 384
[0.011114082299172878, 0.02135222591459751, -0.09249677509069443, 0.06366261094808578, -0.029749928042292595, -0.04083123058080673, 0.06675340235233307, 0.013968830928206444, 0.0006548038218170404, -0.010340639390051365]


In [ ]:
# ============================================================
# STEP 9 - ASTRA DB VECTOR STORE
# ============================================================

vector_store = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="budget_speech_demo",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)

print("Astra DB vector store created")

Astra DB vector store created


In [ ]:
# ============================================================
# STEP 10 - INSERT PDF CHUNKS
# ============================================================

vector_store.add_texts(texts)

print(f"Inserted {len(texts)} chunks into Astra DB")

Inserted 153 chunks into Astra DB


In [ ]:
# ============================================================
# STEP 11 - RETRIEVER
# ============================================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created")

Retriever created


In [ ]:
# ============================================================
# STEP 12 - TEST RETRIEVAL
# ============================================================

query = "What is the current GDP?"

docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} documents")

Retrieved 4 documents


In [ ]:
for i, doc in enumerate(docs, start=1):

    print(f"\n========== DOCUMENT {i} ==========")

    print(doc.page_content[:1000])


========== Document 1 ==========
Revised Estimates 2024-25 
110. The Revised Estimate of the total receipts other than borrowings is  
` 31.47 lakh crore, of which the net tax receipts are ` 25.57 lakh crore. The 
Revised Estimate of the total expenditure is ` 47.16 lakh crore, of which the 
capital expenditure is about ` 10.18 lakh crore. 
111. The Revised Estimate of the fiscal deficit is 4.8 per cent of GDP. 
 19  
 
Budget Estimates 2025-26 
112. Coming to 2025 -26, the total receipts other than  borrowings and the 
total expenditure are estimated at ` 34.96 lakh crore and ` 50.65 lakh crore 
respectively. The net tax receipts are estimated at ` 28.37 lakh crore. 
113. The fiscal deficit is estimated to be 4.4 per cent of GDP. 
114. To finance the fiscal deficit, the net market borrowings from dated

========== Document 2 ==========
decriminalized. Our Government will now bring up the Jan Vishwas Bill 2.0 to 
decriminalize more than 100 provisions in various laws.   
Fiscal Policy

Create the LangChain embedding and LLM objects for later usage:

In [ ]:
# ============================================================
# STEP 13 - HUGGING FACE CLOUD LLM
# ============================================================

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    provider="auto",
    max_new_tokens=512,
    temperature=0.1,
    huggingfacehub_api_token=HF_TOKEN,
)

print("Hugging Face cloud LLM configured")

Create your LangChain vector store ... backed by Astra DB!

In [ ]:
# ============================================================
# STEP 14 - CHAT MODEL
# ============================================================

chat_model = ChatHuggingFace(
    llm=llm
)

print("Hugging Face chat model ready")

In [ ]:
# ============================================================
# STEP 15 - TEST LLM
# ============================================================

response = chat_model.invoke(
    "What is artificial intelligence?"
)

print(response.content)

In [ ]:
# ============================================================
# STEP 16 - RAG PROMPT
# ============================================================

query_text = "What is the current GDP?"

docs = retriever.invoke(query_text)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

prompt = f"""
Answer the question using only the context provided below.

If the answer is not available in the context,
say that you could not find the answer in the PDF.

Context:
{context}

Question:
{query_text}

Answer:
"""

print(prompt)

['GOVERNMENT OF INDIA\nBUDGET 2023-2024\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2023CONTENTS \nPART-A \n Page No.  \n\uf0b7 Introduction 1 \n\uf0b7 Achievements since 2014: Leaving no one behind 2 \n\uf0b7 Vision for Amrit Kaal  – an empowered and inclusive economy 3 \n\uf0b7 Priorities of this Budget 5 \ni. Inclusive Development  \nii. Reaching the Last Mile \niii. Infrastructure and Investment \niv. Unleashing the Potential \nv. Green Growth \nvi. Youth Power  \nvii. Financial Sector  \n \n \n \n \n \n \n \n \n\uf0b7 Fiscal Management 24 \nPART B  \n  \nIndirect Taxes  27 \n\uf0b7 Green Mobility  \n\uf0b7 Electronics   \n\uf0b7 Electrical   \n\uf0b7 Chemicals and Petrochemicals   \n\uf0b7 Marine products  \n\uf0b7 Lab Grown Diamonds  \n\uf0b7 Precious Metals  \n\uf0b7 Metals  \n\uf0b7 Compounded Rubber  \n\uf0b7 Cigarettes  \n  \nDirect Taxes  30 \n\uf0b7 MSMEs and Professionals',
 '\uf0b7 Chemicals and Petrochemicals   \n\uf0b7 Marine products  \n\uf0b7 La

### Load the dataset into the vector store



In [ ]:
# ============================================================
# STEP 17 - GENERATE RAG ANSWER
# ============================================================

response = chat_model.invoke(prompt)

print(response.content)

Inserted 50 headlines.


In [ ]:
# ============================================================
# STEP 18 - INTERACTIVE QA LOOP
# ============================================================

first_question = True

while True:

    if first_question:

        query_text = input(
            "\nEnter your question (or type 'quit' to exit): "
        ).strip()

    else:

        query_text = input(
            "\nWhat's your next question (or type 'quit' to exit): "
        ).strip()


    # --------------------------------------------------------
    # Exit
    # --------------------------------------------------------

    if query_text.lower() == "quit":
        break


    # --------------------------------------------------------
    # Ignore empty questions
    # --------------------------------------------------------

    if query_text == "":
        continue


    first_question = False


    print(f'\nQUESTION: "{query_text}"')


    # ========================================================
    # 1. RETRIEVAL
    # ========================================================

    docs = retriever.invoke(query_text)


    # ========================================================
    # 2. CREATE CONTEXT
    # ========================================================

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )


    # ========================================================
    # 3. CREATE PROMPT
    # ========================================================

    prompt = f"""
Answer the question using only the context provided below.

If the answer is not available in the context,
say that you could not find the answer in the PDF.

Context:
{context}

Question:
{query_text}

Answer:
"""


    # ========================================================
    # 4. CALL HUGGING FACE CLOUD LLM
    # ========================================================

    response = chat_model.invoke(prompt)


    # ========================================================
    # 5. DISPLAY ANSWER
    # ========================================================

    print("\nANSWER:")
    print(response.content)


    # ========================================================
    # 6. DISPLAY RETRIEVED DOCUMENTS
    # ========================================================

    print("\nRELEVANT DOCUMENTS:")

    for i, doc in enumerate(docs, start=1):

        print(f"\n--- Document {i} ---")

        print(doc.page_content[:500])

### Run the QA cycle

Simply run the cells and ask a question -- or `quit` to stop. (you can also stop execution with the "▪" button on the top toolbar)

Here are some suggested questions:
- _What is the current GDP?_
- _How much the agriculture target will be increased to and what the focus will be_


In [ ]:
first_question = True
while True:
    if first_question:
        query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION: \"%s\"" % query_text)
    answer = astra_vector_index.query(query_text, llm=llm).strip()
    print("ANSWER: \"%s\"\n" % answer)

    print("FIRST DOCUMENTS BY RELEVANCE:")
    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print("    [%0.4f] \"%s ...\"" % (score, doc.page_content[:84]))


Enter your question (or type 'quit' to exit): How much the agriculture target will be increased to and what the focus will be

QUESTION: "How much the agriculture target will be increased to and what the focus will be"


ANSWER: "The agriculture credit target will be increased to ` 20 lakh crore with focus on animal husbandry, dairy and fisheries."

FIRST DOCUMENTS BY RELEVANCE:


    [0.9327] "of Millet Research, Hyderabad  will be supported as the Centre of Excellence 
for sh ..."
    [0.9327] "of Millet Research, Hyderabad  will be supported as the Centre of Excellence 
for sh ..."
    [0.9207] "Agriculture Accelerator Fund  
17. An Agriculture Accelerator Fund will be set-up to ..."
    [0.9207] "Agriculture Accelerator Fund  
17. An Agriculture Accelerator Fund will be set-up to ..."

What's your next question (or type 'quit' to exit): What is the current GDP

QUESTION: "What is the current GDP"


ANSWER: "The current GDP is estimated to be 7 per cent."

FIRST DOCUMENTS BY RELEVANCE:


    [0.8920] "estimated to be at 7 per cent. It is notable that this is the highest among all 
the ..."
    [0.8920] "estimated to be at 7 per cent. It is notable that this is the highest among all 
the ..."
    [0.8915] "multiplier impact on growth and employment. After the subdued period of 
the pandemi ..."
    [0.8915] "multiplier impact on growth and employment. After the subdued period of 
the pandemi ..."

What's your next question (or type 'quit' to exit): quit
